In [1]:
import optuna
from sklearn.datasets import load_breast_cancer  # or your own dataset
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import numpy as np


In [2]:
# Load sample classification dataset (replace with your own)
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [17]:
def objective(trial):
    model_name = trial.suggest_categorical("model", [
        "LogisticRegression", 
        "DecisionTreeClassifier",
        "SVC", 
        "GaussianNB",
        "RandomForestClassifier"
    ])

    # Model-specific hyperparameters
    if model_name == "LogisticRegression":
        C = trial.suggest_float("logreg_C", 1e-3, 1e2, log=True)
        model = LogisticRegression(C=C, max_iter=1000)

    elif model_name == "DecisionTreeClassifier":
        max_depth = trial.suggest_int("dtc_max_depth", 2, 20)
        model = DecisionTreeClassifier(max_depth=max_depth)

    elif model_name == "SVC":
        C = trial.suggest_float("svc_C", 1e-3, 1e2, log=True)
        kernel = trial.suggest_categorical("svc_kernel", ["linear", "rbf", "poly"])
        model = SVC(C=C, kernel=kernel)

    elif model_name == "GaussianNB":
        var_smoothing = trial.suggest_float("nb_var_smoothing", 1e-11, 1e-5, log=True)
        model = GaussianNB(var_smoothing=var_smoothing)

    elif model_name == "RandomForestClassifier":
        n_estimators = trial.suggest_int("rf_n_estimators", 50, 200)
        max_depth = trial.suggest_int("rf_max_depth", 2, 20)
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)

    # Optional: wrap in pipeline with StandardScaler for models that need scaling
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    score = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()
    return score


In [20]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

[I 2025-08-05 18:30:48,482] A new study created in memory with name: no-name-78429666-4b53-4dad-ac89-b9b359a22b08
[I 2025-08-05 18:30:49,893] Trial 0 finished with value: 0.9582417582417582 and parameters: {'model': 'RandomForestClassifier', 'rf_n_estimators': 180, 'rf_max_depth': 8}. Best is trial 0 with value: 0.9582417582417582.
[I 2025-08-05 18:30:49,918] Trial 1 finished with value: 0.9692307692307693 and parameters: {'model': 'SVC', 'svc_C': 2.4991664161683067, 'svc_kernel': 'linear'}. Best is trial 1 with value: 0.9692307692307693.
[I 2025-08-05 18:30:49,988] Trial 2 finished with value: 0.9758241758241759 and parameters: {'model': 'LogisticRegression', 'logreg_C': 2.6514064134136306}. Best is trial 2 with value: 0.9758241758241759.
[I 2025-08-05 18:30:50,025] Trial 3 finished with value: 0.9186813186813187 and parameters: {'model': 'DecisionTreeClassifier', 'dtc_max_depth': 19}. Best is trial 2 with value: 0.9758241758241759.
[I 2025-08-05 18:30:50,041] Trial 4 finished with va

In [22]:
print("Best model:", study.best_trial.params["model"])
print("Best score:", study.best_value)
print("Best hyperparameters:", study.best_params)

Best model: LogisticRegression
Best score: 0.9780219780219781
Best hyperparameters: {'model': 'LogisticRegression', 'logreg_C': 2.830103188953982}
